# NeMo fraud detection: data pipeline

Three scripts, chained: inject noise into the raw transcripts, clean and deduplicate them with NeMo Curator, then match the result against ground-truth labels and split into train/validation/test.

```text
transcripts.jsonl -> Noise Injection -> transcripts_noisy.jsonl
  -> NeMo Curator (unicode fix, PII masking, min-length filter, exact dedup)
  -> transcripts_curated.jsonl -> matched against transcripts_benchmark.jsonl
  -> 70/15/15 split -> train.jsonl, validation.jsonl, test.jsonl
```


# 1. Noise injection pipeline

Takes the raw JSONL transcripts and injects artificial defects.

| Defect | Probability |
|---|---:|
| Exact duplicate | 8% |
| PII | 7% |
| Garbage | 5% |
| Encoding error | 3% |
| Clean | ~77% |

The ranges are cumulative:

```text
0.00-0.07  duplicate
0.08-0.14  PII
0.15-0.19  garbage
0.20-0.22  encoding
0.23-0.99  clean
```

`random.random()` means the exact split varies run to run.


In [ ]:
import sys
import json
import random
import os
from pathlib import Path
from typing import Any, Dict, List

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd().resolve().parents[1]))
DATA_DIR = PROJECT_ROOT / "notebooks" / "01_Data_Generation" / "data"

ORIGINAL_RAW_PATH = DATA_DIR / "transcripts.jsonl"
NOISY_RAW_PATH = DATA_DIR / "transcripts_noisy.jsonl"

NOISY_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)

PROB_DUPLICATE = 0.08
PROB_PII = 0.07
PROB_GARBAGE = 0.05
PROB_ENCODING = 0.03


## 1.1 Paths and configuration

`Path` holds the file locations. `mkdir(..., exist_ok=True)` makes sure the target directory exists.

The four `PROB_*` variables control the noise injection; whatever probability mass is left over is the clean share.


In [ ]:
PII_SAMPLES = [
    "Meine Kreditkartennummer lautet 4532-8921-1029-4411 mit CVV 892.",
    "Sie erreichen mich unter max.mustermann@example.de oder Mobil: +49 171 1234567.",
    "Meine IBAN lautet DE89 3704 0044 0532 0130 00, Inhaber ist Thomas Müller.",
    "Ich wohne in der Hauptstraße 45, 10115 Berlin. Geburtsdatum ist der 14.05.1982.",
    "Meine Sozialversicherungsnummer ist 12 140582 M 043."
]

GARBAGE_SNIPPETS = [
    "???", "N/A", "asdfghjkl;", "NULL",
    "[ERROR_VOICEMAIL_RECORDING_CORRUPTED]",
    "CLICK... BEEP... BEEP...", "a"
]


## 1.2 PII and garbage samples

`PII_SAMPLES` stands in for sensitive data (kept in German, matching the transcripts). `GARBAGE_SNIPPETS` stands in for corrupted or empty text.

The curator pipeline is expected to catch both of these back out.


In [ ]:
def load_jsonl(file_path: Path) -> List[Dict[str, Any]]:
    records = []
    if not file_path.exists():
        return records

    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line_str = line.strip()
            if not line_str:
                continue
            try:
                records.append(json.loads(line_str))
            except json.JSONDecodeError as e:
                print(f"WARNING: invalid JSON on line {line_no}: {e}")
    return records


def format_to_text(item: Dict[str, Any]) -> str:
    if "text" in item and isinstance(item["text"], str):
        return item["text"]

    raw_response = item.get("response", "")
    if not raw_response:
        return ""

    try:
        parsed = json.loads(raw_response) if isinstance(raw_response, str) else raw_response
        if isinstance(parsed, dict) and "transcript" in parsed:
            return "\n".join(
                f"{t.get('speaker', 'Unknown')}: {t.get('text', '')}"
                for t in parsed["transcript"]
            )
    except (json.JSONDecodeError, TypeError):
        pass

    return ""


## 1.3 Load JSONL and extract text

`load_jsonl()` reads the file line by line; an invalid JSON line prints a warning and is skipped.

`format_to_text()` handles two input shapes: a plain `text` field, or a `response` field holding a `transcript` list, so the script normalizes either raw format to one text string.


In [ ]:
def main():
    print(f"Loading original transcripts from: {ORIGINAL_RAW_PATH}")
    input_records = load_jsonl(ORIGINAL_RAW_PATH)

    if not input_records:
        print(
            f"FAILED: no data found in '{ORIGINAL_RAW_PATH}', "
            "or the file does not exist. Aborting."
        )
        sys.exit(1)

    print(
        f"Injecting garbage, PII, duplicates and encoding errors "
        f"into {len(input_records)} records..."
    )

    noisy_dataset = []
    stats = {"duplicate": 0, "pii": 0, "garbage": 0, "encoding": 0, "clean": 0}

    for idx, item in enumerate(input_records):
        call_id = item.get("call_id", item.get("id", f"CALL_{idx:04d}"))
        flat_text = format_to_text(item)

        if not flat_text:
            continue

        dice = random.random()

        if dice < PROB_DUPLICATE:
            noisy_dataset.append({
                "call_id": call_id,
                "text": flat_text,
                "is_clean": True,
                "noise_type": "none"
            })
            noisy_dataset.append({
                "call_id": f"{call_id}_DUP",
                "text": flat_text,
                "is_clean": False,
                "noise_type": "exact_duplicate"
            })
            stats["duplicate"] += 1

        elif dice < (PROB_DUPLICATE + PROB_PII):
            pii_text = f"{flat_text}\nAnrufer: {random.choice(PII_SAMPLES)}"
            noisy_dataset.append({
                "call_id": call_id,
                "text": pii_text,
                "is_clean": False,
                "noise_type": "pii_injection"
            })
            stats["pii"] += 1

        elif dice < (PROB_DUPLICATE + PROB_PII + PROB_GARBAGE):
            noisy_dataset.append({
                "call_id": call_id,
                "text": random.choice(GARBAGE_SNIPPETS),
                "is_clean": False,
                "noise_type": "garbage_text"
            })
            stats["garbage"] += 1

        elif dice < (
            PROB_DUPLICATE + PROB_PII + PROB_GARBAGE + PROB_ENCODING
        ):
            broken_text = (
                flat_text
                .replace("ä", "Ã¤")
                .replace("ö", "Ã¶")
                .replace("ü", "Ã¼")
                .replace("ß", "Ã\x9f")
            )
            noisy_dataset.append({
                "call_id": call_id,
                "text": broken_text,
                "is_clean": False,
                "noise_type": "encoding_error"
            })
            stats["encoding"] += 1

        else:
            noisy_dataset.append({
                "call_id": call_id,
                "text": flat_text,
                "is_clean": True,
                "noise_type": "none"
            })
            stats["clean"] += 1

    print(f"Writing the noisy test data to: {NOISY_RAW_PATH}")
    with open(NOISY_RAW_PATH, "w", encoding="utf-8") as f:
        for rec in noisy_dataset:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print("\nINJECTION STATISTICS:")
    print(f"  - Original: {len(input_records)}")
    print(f"  - Duplicates: {stats['duplicate']}")
    print(f"  - PII: {stats['pii']}")
    print(f"  - Garbage: {stats['garbage']}")
    print(f"  - Encoding: {stats['encoding']}")
    print(f"  - Clean: {stats['clean']}")
    print(f"\n{len(noisy_dataset)} records written.")


if __name__ == "__main__":
    main()


### 1.4 Core noise logic

A random number per record decides which defect applies.

**Important:** the duplicate case writes two records, the original plus a `_DUP` copy, so the output file can hold more rows than the input. Noise injection and the curator pipeline are deliberately paired: this step creates the exact duplicates and encoding errors that the next step is meant to catch and remove.


# 2. NeMo Curator pipeline

Cleans `transcripts_noisy.jsonl`:

```text
schema validation -> NeMo Curator dataset -> add ID -> unicode fix
  -> PII masking -> min-length filter -> exact dedup (MD5)
  -> transcripts_curated.jsonl
```


In [ ]:
import sys
import json
import re
import shutil
from pathlib import Path
from typing import Any
import os

from nemo_curator import Modify, ScoreFilter, Sequential, AddId
from nemo_curator.datasets import DocumentDataset
from nemo_curator.filters import DocumentFilter
from nemo_curator.modules import ExactDuplicates
from nemo_curator.utils.distributed_utils import get_client
from nemo_curator.modifiers import DocumentModifier, UnicodeReformatter

from dask_cuda import LocalCUDACluster
from distributed import Client


## 2.1 GPU/CUDA setup

NeMo Curator starts with a GPU/Dask backend. `LocalCUDACluster` and `Client` are imported but `main()` does not use them directly.

The three environment variables set here work around cuFile/GDS issues in containerized environments.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd().resolve().parents[1]))
DATA_DIR = PROJECT_ROOT / "notebooks"

INPUT_PATH = DATA_DIR / "01_Data_Generation" / "data" / "transcripts_noisy.jsonl"
CURATED_OUT_PATH = DATA_DIR / "02_Data_Curation" / "data" / "transcripts_curated.jsonl"
TEMP_EXPORT_DIR = DATA_DIR / "02_Data_Curation" / "data" / "_temp_curator_export"
DEDUP_LOG_DIR = DATA_DIR / "02_Data_Curation" / "data" / "dedup_logs"
DEDUP_CACHE_DIR = DATA_DIR / "02_Data_Curation" / "data" / "dedup_cache"

CURATED_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
DEDUP_LOG_DIR.mkdir(parents=True, exist_ok=True)
DEDUP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Workaround for cuFile/GDS issues in containers
os.environ["KVIKIO_COMPAT_MODE"] = "ON"
os.environ["CUDF_CUFILE_ENABLED"] = "0"
os.environ["RAPIDS_NO_CUFILE"] = "1"

print("Paths and CUDA environment ready.")
print(f"Expected input file: {INPUT_PATH}")


## 2.2 Strict input validation

`validate_input_file()` fails fast: it aborts if the file is missing, the JSON is invalid, the `text` field is absent, or the file is empty, so errors surface before the GPU/Curator pipeline starts.


In [ ]:
def validate_input_file(file_path: Path) -> int:
    print(f"Validating input file & schema: {file_path.name}...")

    if not file_path.exists():
        print(f"FATAL: '{file_path}' does not exist.")
        sys.exit(1)

    total_lines = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line_str = line.strip()
            if not line_str:
                continue

            try:
                record = json.loads(line_str)
            except json.JSONDecodeError as e:
                print(
                    f"SCHEMA ERROR [line {line_no}]: "
                    f"not valid JSON. Error: {e}"
                )
                sys.exit(1)

            if "text" not in record:
                print(
                    f"SCHEMA ERROR [line {line_no}]: "
                    "required field 'text' is missing."
                )
                sys.exit(1)

            total_lines += 1

    if total_lines == 0:
        print("FATAL: input file is completely empty.")
        sys.exit(1)

    print(f"Validation passed: {total_lines} records.")
    return total_lines


## 2.3 PII masking

`FraudPiiModifier` is a custom NeMo Curator extension. It uses regular expressions to replace:

- IBANs
- credit card numbers
- email addresses
- dates of birth
- mobile numbers
- street addresses
- customer numbers

A recognized IBAN becomes `[IBAN_MASKIERT]`, for example. Detection is pattern-based: only strings matching the regex get masked, so a format the regex doesn't anticipate passes through.


In [ ]:
class FraudPiiModifier(DocumentModifier):
    def modify_document(self, doc: str) -> str:
        if not isinstance(doc, str):
            return doc

        doc = re.sub(
            r'DE\d{2}\s?(\d{4}\s?){4}\d{2}',
            '[IBAN_MASKIERT]', doc
        )

        doc = re.sub(
            r'\b(?:\d[ -]*?){13,16}\b',
            '[KREDITKARTE_MASKIERT]', doc
        )

        doc = re.sub(
            r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
            '[EMAIL_MASKIERT]', doc
        )

        doc = re.sub(
            r'\b\d{1,2}\.\s+'
            r'(?:Januar|Februar|März|April|Mai|Juni|Juli|August|'
            r'September|Oktober|November|Dezember)\s+\d{4}\b',
            '[GEBURTSDATUM_MASKIERT]', doc, flags=re.IGNORECASE
        )

        doc = re.sub(
            r'\b(?:\+49|0)\s*1[567]\d[\s\-]?\d{3,8}\b',
            '[HANDYNUMMER_MASKIERT]', doc
        )

        doc = re.sub(
            r'\b(?:[A-ZÄÖÜ][a-zäöüß]+\s+)?'
            r'[A-ZÄÖÜ][a-zäöüß]+'
            r'(?:straße|str\.|weg|allee|platz|ring)\s+\d+[a-zA-Z]?,?'
            r'\s*\d{5}\s+[A-ZÄÖÜ][a-zäöüß]+\b',
            '[ADRESSE_MASKIERT]', doc, flags=re.IGNORECASE
        )

        doc = re.sub(
            r'\b\d{9}\b',
            '[KUNDENNUMMER_MASKIERT]', doc
        )

        return doc


## 2.4 Minimum length filter

`MinLengthFilter` measures `len(str(text).strip())`. `keep_document()` returns `True` only at 50 characters or more, and accepts a dict, an object with a text attribute, or a plain string.


In [ ]:
class MinLengthFilter(DocumentFilter):
    def __init__(self, min_length: int = 50, text_field: str = "text"):
        super().__init__()
        self.min_length = min_length
        self.text_field = text_field

    def score_document(self, doc: Any) -> int:
        if isinstance(doc, dict):
            text = doc.get(self.text_field, "")
        elif hasattr(doc, self.text_field):
            text = getattr(doc, self.text_field, "")
        elif isinstance(doc, str):
            text = doc
        else:
            return 0

        return len(str(text).strip())

    def keep_document(self, score: int) -> bool:
        return score >= self.min_length


In [ ]:
import os
os.environ["DASK_DATAFRAME__QUERY_PLANNING"] = "False"
def main():
    initial_count = validate_input_file(INPUT_PATH)

    print("Initializing the NeMo Curator execution client (GPU/CUDA backend)...")
    client = get_client(cluster_type="gpu", set_torch_to_use_rmm=False)
    print("Dask client connected.")

    print("1. Loading the dataset into NeMo Curator...")
    dataset = DocumentDataset.read_json(str(INPUT_PATH), add_filename=True, backend="pandas")

    print("2. Generating unique IDs (required for deduplication)...")
    add_id = AddId(id_field="id", id_prefix="FRAUD_data", start_index=0)
    dataset = add_id(dataset)

    print("3. Running the cleaning sequence (unicode fix via ftfy, PII masking)...")
    cleaners = Sequential([
        Modify(UnicodeReformatter()),
        Modify(FraudPiiModifier())
    ])
    dataset = cleaners(dataset).persist()

    print("4. Applying the minimum-length filter (50 characters)...")
    length_filter = ScoreFilter(
        MinLengthFilter(min_length=50, text_field="text"),
        score_type=int
    )
    dataset = length_filter(dataset)

    print("5. Running exact deduplication...")
    exact_dup = ExactDuplicates(
        logger=str(DEDUP_LOG_DIR),
        id_field="id",
        text_field="text",
        hash_method="md5",
        cache_dir=str(DEDUP_CACHE_DIR),
    )
    duplicates_dataset = exact_dup(dataset=dataset)
    
    # Identify the duplicates to remove
    exact_docs_to_remove = duplicates_dataset.df.map_partitions(
        lambda x: x[x._hashes.duplicated(keep="first")]
    )

    # Apply the filter
    id_field = "id"
    cleaned_df = dataset.df[
        ~dataset.df[id_field].isin(exact_docs_to_remove[id_field].compute())
    ]
    dataset = DocumentDataset(cleaned_df)

    # Clean up old temp folders and old output files
    if TEMP_EXPORT_DIR.exists():
        shutil.rmtree(TEMP_EXPORT_DIR)
    if CURATED_OUT_PATH.exists():
        if CURATED_OUT_PATH.is_dir():
            shutil.rmtree(CURATED_OUT_PATH)
        else:
        
            CURATED_OUT_PATH.unlink()

    print(f"Writing the final curated dataset to: {CURATED_OUT_PATH}")
    dataset.to_json(str(TEMP_EXPORT_DIR), write_to_filename=False)

    # Merge the partition files
    exported_files = list(TEMP_EXPORT_DIR.glob("*.json*")) + list(TEMP_EXPORT_DIR.glob("*.part"))
    if exported_files:
        shutil.move(str(exported_files[0]), str(CURATED_OUT_PATH))
        shutil.rmtree(TEMP_EXPORT_DIR)
    else:
        with open(CURATED_OUT_PATH, "w", encoding="utf-8") as outfile:
            for part in sorted(TEMP_EXPORT_DIR.iterdir()):
                if part.is_file():
                    with open(part, "r", encoding="utf-8") as infile:
                        shutil.copyfileobj(infile, outfile)
        shutil.rmtree(TEMP_EXPORT_DIR)

    # Final tally
    final_docs = []
    with open(CURATED_OUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                final_docs.append(json.loads(line))
    
    final_count = len(final_docs)
    total_removed = initial_count - final_count

    print("\n" + "="*80)
    print("NEMO CURATOR PIPELINE STATISTICS")
    print("="*80)
    print(f"  Raw records read:          {initial_count}")
    print(f"  Filtered / duplicate removed: {total_removed}")
    print("--------------------------------------------------------------------------------")
    print(f"  Curated records remaining: {final_count}")
    print(f"  Total discard rate:        {(total_removed / initial_count):.2%}")
    print("="*80)
    print(f"\nPipeline finished. Output file: {CURATED_OUT_PATH.name}")

if __name__ == "__main__":
    main()

# 3. SFT data preparation

Matches curated text to ground-truth labels and splits the result.

```text
transcripts_curated.jsonl --id--> match <-- transcripts_benchmark.jsonl
  -> {"input": text, "output": label} -> shuffle (seed 42) -> 70/15/15
  -> train.jsonl, validation.jsonl, test.jsonl
```

Note: the split only checks that train/validation/test add up correctly. It does not check the fraud/legitimate balance within each split.


In [ ]:
import sys
import json
import random
import os
from pathlib import Path

PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd().resolve().parents[1]))
DATA_DIR = PROJECT_ROOT / "notebooks"

CURATED_PATH = DATA_DIR / "02_Data_Curation" / "data" / "transcripts_curated.jsonl"
BENCHMARK_PATH = DATA_DIR / "01_Data_Generation" / "data" / "benchmark.jsonl"
SFT_DIR = DATA_DIR / "02_Data_Curation" / "data" / "sft"


## 3.1 Fail-fast and JSONL helpers

Checks both input files exist before doing anything else. `load_jsonl()` reads records; `save_jsonl()` writes them back out, one JSON object per line.

Matching runs on `call_id` or `id` from the curated file against the `id` key in the ground-truth map built from the benchmark file. A mismatch between those IDs produces an unmatched (dropped) record.


In [ ]:
def validate_required_files():
    print("Checking input files...")

    if not CURATED_PATH.exists():
        print(f"FAILED: curated file not found: '{CURATED_PATH}'")
        sys.exit(1)

    if not BENCHMARK_PATH.exists():
        print(f"FAILED: benchmark file not found: '{BENCHMARK_PATH}'")
        sys.exit(1)

    print("All required files found.\n")


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


In [ ]:
def main():
    validate_required_files()

    print("Preparing SFT datasets...")
    
    curated_data = load_jsonl(CURATED_PATH)
    benchmark_data = load_jsonl(BENCHMARK_PATH)

    # 1. Build a lookup map from the benchmark file (keyed by "id")
    gt_map = {}
    for item in benchmark_data:
        cid = item.get("id")
        label = item.get("label") or item.get("fraud_type")
        if cid and label:
            gt_map[str(cid)] = label

    # 2. Match on 'call_id' or 'id'
    sft_samples = []
    matched_count = 0
    unmatched_count = 0

    for doc in curated_data:
        text = doc.get("text", "")
        cid = str(doc.get("call_id") or doc.get("id", ""))
        
        gt_label = gt_map.get(cid)
        
        if gt_label:
            matched_count += 1
            sft_samples.append({
                "input": text,
                "output": gt_label
            })
        else:
            unmatched_count += 1

    total_docs = len(curated_data)
    print(f"Curated documents loaded: {total_docs}")
    print(f"    Matched to ground truth: {matched_count}")
    
    if unmatched_count > 0:
        print(f"    Skipped, no matching label: {unmatched_count} document(s)")

    if len(sft_samples) == 0:
        print(f"\nFATAL: no SFT records could be matched at all.")
        sys.exit(1)

    # 3. Train/validation/test split (70% / 15% / 15%)
    random.seed(42)
    random.shuffle(sft_samples)

    total_samples = len(sft_samples)
    train_end = int(total_samples * 0.70)
    val_end = train_end + int(total_samples * 0.15)

    train_data = sft_samples[:train_end]
    val_data = sft_samples[train_end:val_end]
    test_data = sft_samples[val_end:]

    # 4. Save
    SFT_DIR.mkdir(parents=True, exist_ok=True)
    
    train_path = SFT_DIR / "train.jsonl"
    val_path = SFT_DIR / "validation.jsonl"
    test_path = SFT_DIR / "test.jsonl"

    save_jsonl(train_data, train_path)
    save_jsonl(val_data, val_path)
    save_jsonl(test_data, test_path)

    print(f"\n    -> Wrote train.jsonl ({len(train_data)} records)")
    print(f"    -> Wrote validation.jsonl ({len(val_data)} records)")
    print(f"    -> Wrote test.jsonl ({len(test_data)} records)")
    print(f"\nSFT datasets saved to '{SFT_DIR}'.")

if __name__ == "__main__":
    main()


# 4. Example: one record through the pipeline

**Raw:**
```json
{"id": "doc-00001", "text": "Kunde: Meine Karte funktioniert nicht ..."}
```

**After noise injection** (PII appended):
```text
Kunde: Meine Karte funktioniert nicht ...
Anrufer: Meine IBAN lautet ...
```

**After curation**, the IBAN becomes `[IBAN_MASKIERT]`. A very short text or an exact duplicate would be dropped here instead.

**After SFT preparation**, matched to its label:
```json
{
  "input": "Kunde: Meine Karte funktioniert nicht ...",
  "output": "legitimate"
}
```
